# PySpark — Joins

Joins combine two DataFrames on a common key.

```python
df1.join(df2, on="key", how="inner")
```

| Join Type | Keeps rows from... | Missing side becomes |
|-----------|-------------------|---------------------|
| `inner` (default) | Both sides match | — (rows dropped) |
| `left` / `left_outer` | All of left, matching right | `null` |
| `right` / `right_outer` | All of right, matching left | `null` |
| `outer` / `full` / `full_outer` | All rows from both | `null` |
| `left_semi` | Left rows that have a match | No right columns included |
| `left_anti` | Left rows with **no** match | No right columns included |
| `cross` | All combinations (cartesian) | — |

## Setup — Two Related DataFrames

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("Joins") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

employees = spark.createDataFrame([
    (1, "Alice",   "Eng",  95000),
    (2, "Bob",     "Mkt",  72000),
    (3, "Charlie", "Eng",  110000),
    (4, "Diana",   "HR",   65000),
    (5, "Eve",     "Fin",  88000),
    (6, "Frank",   None,   75000),
], ["emp_id", "name", "dept_code", "salary"])

departments = spark.createDataFrame([
    ("Eng",  "Engineering",     "New York"),
    ("Mkt",  "Marketing",       "California"),
    ("HR",   "Human Resources", "Texas"),
    ("Leg",  "Legal",           "New York"),
], ["dept_code", "dept_name", "location"])

print("=== Employees ===")
employees.show()
print("=== Departments ===")
departments.show()

## 1. INNER JOIN — Only Matching Rows

Keeps rows where the join key exists in **both** DataFrames.  
Eve (Fin) and Frank (None) are dropped. Legal department is dropped.

In [ ]:
inner = employees.join(departments, on="dept_code", how="inner")
inner.show()
print("Row count:", inner.count())   # 4 — Eve and Frank dropped, Legal dropped

## 2. LEFT JOIN — All Left Rows, Matching Right

Keeps **all** rows from the left (employees) DataFrame.  
If no match in right, right columns are `null`.  
Eve and Frank are kept; Legal department is still dropped.

In [ ]:
left = employees.join(departments, on="dept_code", how="left")
left.show()
print("Row count:", left.count())   # 6 — all employees kept, Eve/Frank have null dept info

## 3. RIGHT JOIN — All Right Rows, Matching Left

Keeps **all** rows from the right (departments) DataFrame.  
Legal gets a row; Frank and Eve are dropped.

In [ ]:
right = employees.join(departments, on="dept_code", how="right")
right.show()
print("Row count:", right.count())   # 5 — all departments kept, Legal has null employee info

## 4. FULL OUTER JOIN — All Rows from Both Sides

Everything appears. Non-matching rows get `null` on the side they have no partner.

In [ ]:
full = employees.join(departments, on="dept_code", how="full")
full.orderBy("emp_id").show()
print("Row count:", full.count())   # 7 — everyone appears

## 5. LEFT SEMI — Existence Filter (No Right Columns)

Like an INNER JOIN but returns **only left columns** — useful to filter "which employees have a department?"

In [ ]:
semi = employees.join(departments, on="dept_code", how="left_semi")
semi.show()   # Only employees who matched — no dept columns

## 6. LEFT ANTI — Find Non-Matches

The opposite of semi — returns left rows with **no match** in the right.  
Useful for finding orphan records or data quality issues.

In [ ]:
anti = employees.join(departments, on="dept_code", how="left_anti")
anti.show()   # Eve (Fin) and Frank (None) — no matching department

# Also useful: which departments have NO employees?
dept_anti = departments.join(employees, on="dept_code", how="left_anti")
dept_anti.show()   # Legal — no employees

## 7. Joining on Multiple Columns

In [ ]:
# When the join key has a different name in each DataFrame
# Use a join condition instead of a string
employees.join(
    departments,
    on=employees["dept_code"] == departments["dept_code"],
    how="inner"
).drop(departments["dept_code"]).show()   # drop duplicate key column

# Multi-column join
df1 = spark.createDataFrame([(1, 2023, 100), (2, 2023, 200)], ["id", "year", "amount"])
df2 = spark.createDataFrame([(1, 2023, "A"), (2, 2022, "B")], ["id", "year", "grade"])
df1.join(df2, on=["id", "year"], how="inner").show()   # matches on BOTH id AND year

## 8. Real-World: Employee Enrichment Pipeline

In [ ]:
# Enrich employee records with department info, then build a report
enriched = (
    employees
    .join(departments, on="dept_code", how="left")
    .select(
        col("emp_id"),
        col("name"),
        col("dept_name"),
        col("location"),
        col("salary")
    )
    .orderBy("dept_name", col("salary").desc())
)
enriched.show()

# Find employees missing department assignment
no_dept = employees.join(departments, on="dept_code", how="left_anti")
print("\nEmployees with no valid department:")
no_dept.show()

## Quick Reference

```python
# String key (same name in both DFs)
df1.join(df2, on="key", how="inner")
df1.join(df2, on=["key1", "key2"], how="left")

# Condition (different key names, or complex condition)
df1.join(df2, on=df1["id"] == df2["emp_id"], how="right")

# Join types
how="inner"        # only matching rows
how="left"         # all left + matching right
how="right"        # all right + matching left
how="outer"        # all rows from both
how="full"         # same as outer
how="left_semi"    # left rows that have a match (no right columns)
how="left_anti"    # left rows that have NO match
how="cross"        # every combination (cartesian product)
```

## Interview Questions

1. What is the difference between inner join and left join?
2. When would you use a `left_anti` join? Give a real-world example.
3. What is the difference between `left_semi` and inner join?
4. What happens to rows that don't match in a full outer join?
5. How do you join on a column that has different names in each DataFrame?
6. How do you prevent duplicate column names after a join?
7. What is a cartesian/cross join? When is it useful and when is it dangerous?

## Practice Exercises

**Easy:**
1. Inner join `employees` with `departments` — show name, dept_name, salary.
2. Left join — show all employees, with `null` where department is missing.
3. Count how many employees are in each location after joining.

**Medium:**
4. Find all employees whose department code doesn't exist in the departments table.
5. Which departments have at least one employee? (Use `left_semi` on departments side.)
6. Enrich employees with location, then find the average salary per location.

**Advanced:**
7. Self-join: find all pairs of employees in the same department.
8. Build a salary comparison: join with a `budget` table (dept_code, budget_amount) and flag departments over budget.

In [ ]:
spark.stop()